# Notebook 6 — Generazione del modello bayesiano source (digits, SVHN)

Stessa filosofia del Notebook 2 (`f = h ∘ g`, solo `h` riceve il
trattamento bayesiano), applicata qui a SVHN come source invece di MNIST:
SVHN (foto reali di numeri civici) è scelto per lo shift più marcato verso
MNIST/USPS (scansioni pulite) -- uno scenario di adattamento più
informativo di MNIST↔USPS, il cui source-only è già troppo alto per
lasciare margine a un adattamento di mostrare un effetto.

Il training vero e proprio (MAP + early stopping, necessario per SVHN --
diversamente da MNIST nel Notebook 2, che converge in 3 epoche fisse con
`train_map`, SVHN overfitta se non fermato su una validazione) vive in
`src/digits_train.py`, non qui. La cella di setup sotto lo lancia
automaticamente **solo se manca ancora un checkpoint** (~15-25 minuti su
MPS); se il checkpoint esiste già, lo salta e stampa un avviso, senza
riaddestrare.

Per verificare esplicitamente che il checkpoint ricaricato riproduca
l'accuracy riportata a fine training (fatto una volta, non ripetuto qui):

```bash
python src/digits_verify.py
```

Questo notebook carica il checkpoint e fa il fit di Laplace sull'ultimo
layer -- stesso schema di come i Notebook 3 e 4 caricano gli artefatti
salvati dal Notebook 2.

## Setup

In [6]:
import sys
from pathlib import Path

cwd = Path().resolve()
PROJ = cwd
while not (PROJ / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from src.digits_data import load_domain
from src.digits_model import SmallCNN32
from src.bayesian_model import extract, head_weights, augment, LastLayerLaplace

CHECKPOINT_PATH = PROJ / "models" / "source_svhn" / "model.pt"
SOURCE_DOMAIN = "svhn"
TARGET_DOMAINS = ["mnist", "usps"]
BATCH_SIZE = 128

if not CHECKPOINT_PATH.exists():
    print(f"{CHECKPOINT_PATH} non trovato -- lancio il training (src/digits_train.py)...")
    ROOT_DIR = PROJ.parent
    sys.path.insert(0, str(ROOT_DIR))
    from code_v2.src.digits_train import main as train_digits_main
    train_digits_main()
else:
    print(f"{CHECKPOINT_PATH} trovato -- training saltato")

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
mean, std = ckpt["source_mean"], ckpt["source_std"]
WEIGHT_DECAY = ckpt["weight_decay"]
N_SOURCE_TRAIN = ckpt["n_source_train"]
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"source_mean={mean:.4f} source_std={std:.4f} weight_decay={WEIGHT_DECAY} "
      f"n_source_train={N_SOURCE_TRAIN}")

/Users/riccardo/Desktop/PML/BayesianExoAdaptation/code_v2/models/source_svhn/model.pt non trovato -- lancio il training (src/digits_train.py)...
Device: mps
Computing svhn (source) normalization stats from its training split...
  mean=0.4453  std=0.1970
svhn train: 73257 images   svhn test: 26032 images
Internal split (SVHN train only): 65932 train / 7325 val (val_fraction=0.1, seed=2019) -- SVHN test untouched until the final report

Training configuration:
  optimizer=AdamW  lr=0.001  weight_decay=0.001
  batch_size=128  max_epochs=30  patience=5
epoch   1/30: train_loss=0.5684 train_acc=0.8401  val_loss=0.5836 val_acc=0.8315
epoch   2/30: train_loss=0.4436 train_acc=0.8731  val_loss=0.4758 val_acc=0.8599
epoch   3/30: train_loss=0.3775 train_acc=0.8917  val_loss=0.4255 val_acc=0.8758
epoch   4/30: train_loss=0.3377 train_acc=0.9051  val_loss=0.4037 val_acc=0.8780
epoch   5/30: train_loss=0.2942 train_acc=0.9150  val_loss=0.3806 val_acc=0.8848
epoch   6/30: train_loss=0.2637 train_ac

## 1. Caricamento del source model verificato + domini target

`src/digits_verify.py` ha già confermato che questo checkpoint, ricaricato
da zero, riproduce esattamente l'accuracy riportata a fine training (PASS
su SVHN test, MNIST test e USPS test). Qui viene ricaricato allo stesso
modo, insieme a MNIST e USPS come dataset separati per le sezioni
successive.

In [7]:
model = SmallCNN32(n_classes=ckpt["n_classes"], feature_dim=ckpt["feature_dim"])
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(f"model.training = {model.training} (deve essere False)")

target_data = {}
for domain in TARGET_DOMAINS:
    X_t, y_t = load_domain(domain, "test", mean, std)
    target_data[domain] = (X_t, y_t)
    print(f"dominio target '{domain}': {X_t.shape[0]} immagini (test split, "
          f"normalizzate con le statistiche di {SOURCE_DOMAIN}-train)")

model.training = False (deve essere False)
dominio target 'mnist': 10000 immagini (test split, normalizzate con le statistiche di svhn-train)
dominio target 'usps': 2007 immagini (test split, normalizzate con le statistiche di svhn-train)


## 2. Fit di Laplace sull'ultimo layer

`tau_prior = weight_decay * N_source`: `weight_decay` è quello che
`src/digits_train.py` ha effettivamente passato ad `AdamW` (letto dal
checkpoint, non assunto); `N_source = ckpt["n_source_train"]` è la
dimensione dello split di training interno che l'ottimizzatore ha
effettivamente visto (esclude lo split di validazione per l'early
stopping).

Il fit gira sull'**intero** SVHN train (nessun sottocampionamento):
`weight_space_hessian` (`laplace_core.py`) costruisce l'Hessiana blocco per
blocco via BLAS (`K²` blocchi `D×D`), senza mai materializzare un tensore
`(N,Dp,Dp)` -- a differenza dell'implementazione usata in una versione
precedente di questo esperimento, che per lo stesso motivo doveva
sottocampionare SVHN train a poche migliaia di immagini.

In [8]:
tau_prior = WEIGHT_DECAY * N_SOURCE_TRAIN
print(f"tau_prior = weight_decay * N_source = {WEIGHT_DECAY} * {N_SOURCE_TRAIN} = {tau_prior:.3f}")

X_svhn_train, y_svhn_train = load_domain(SOURCE_DOMAIN, "train", mean, std)
print(f"{SOURCE_DOMAIN} train (pool designato per il fit): {X_svhn_train.shape[0]} immagini")

train_loader = DataLoader(TensorDataset(X_svhn_train, y_svhn_train), batch_size=256, shuffle=False)
Phi, y_np, _ = extract(model, train_loader, device="cpu")
Phi_aug = augment(Phi)
W_aug = head_weights(model)
print(f"Phi: {Phi.shape}  Phi_aug: {Phi_aug.shape}  W_aug: {W_aug.shape}")

laplace = LastLayerLaplace.fit(W_aug, Phi_aug, tau_prior=tau_prior)
print(f"\nLaplace fit: K={laplace.K}  Dp={laplace.Dp}  cov shape={laplace.cov.shape}")

map_preds = (Phi_aug @ W_aug.T).argmax(axis=1)
map_acc = (map_preds == y_np).mean()
print(f"sanity check -- MAP accuracy sulle stesse {len(y_np)} feature di training: {100 * map_acc:.2f}%")

MODELS_DIR = CHECKPOINT_PATH.parent
np.savez(MODELS_DIR / "svhn_laplace.npz", theta_map=laplace.theta_map, cov=laplace.cov,
        K=laplace.K, Dp=laplace.Dp, tau_prior=tau_prior, source_mean=mean, source_std=std)
print(f"\nsalvato {MODELS_DIR / 'svhn_laplace.npz'}")

tau_prior = weight_decay * N_source = 0.001 * 65932 = 65.932
svhn train (pool designato per il fit): 73257 immagini
Phi: (73257, 128)  Phi_aug: (73257, 129)  W_aug: (10, 129)

Laplace fit: K=10  Dp=129  cov shape=(1290, 1290)
sanity check -- MAP accuracy sulle stesse 73257 feature di training: 93.54%

salvato /Users/riccardo/Desktop/PML/BayesianExoAdaptation/code_v2/models/source_svhn/svhn_laplace.npz


## Confronto pulito source vs. target

A differenza di una versione precedente di questo esperimento (dove il
checkpoint source arrivava già pre-addestrato, senza uno split
disponibile, e ogni valutazione "source" era in realtà sui punti di
training), qui `svhn_train.npz` (usato per il fit sopra) e `svhn_test.npz`
(usato più avanti come baseline source) sono due file disgiunti: la
baseline source nei prossimi notebook è quindi pulita, su dati mai visti
né dal training né dal fit -- confermato di seguito controllando
esplicitamente quale split è stato usato dove.

In [9]:
X_svhn_test, y_svhn_test = load_domain(SOURCE_DOMAIN, "test", mean, std)
print(f"fit (sopra):              digits/svhn_train.npz  ({X_svhn_train.shape[0]} immagini)")
print(f"baseline source (dopo):   digits/svhn_test.npz   ({X_svhn_test.shape[0]} immagini)")
print("file disgiunti, caricati indipendentemente -- non possono sovrapporsi")

fit (sopra):              digits/svhn_train.npz  (73257 immagini)
baseline source (dopo):   digits/svhn_test.npz   (26032 immagini)
file disgiunti, caricati indipendentemente -- non possono sovrapporsi
